# ComplaintIQ - first look at the data

Get the environment working, load the ETL's Parquet, and run a standard first pass: peek, shape, types, summary, missing values, and target imbalance.

The exploration splits by task:
- **`02_eda_supervised.ipynb`** - supervised targets (relief risk, ranking, product/theme)
- **`03_eda_unsupervised.ipynb`** - unsupervised work (clustering, similar-complaint retrieval)

## EDA objectives

Fast confidence check before the deeper notebooks:
- Is the environment (Spark + seaborn) wired up and reproducible?
- Does the ETL's Parquet load and match the schema?
- Where are the missing values? How imbalanced is `monetary_relief`?

Fix the data here if anything looks wrong. Don't move to `02` / `03` until it does.

> **Note:** Spark does all reading and counting. We only `.toPandas()` small results.

> **Go deeper:**
> - [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/): source dataset and collection.
> - [PySpark: DataFrame quickstart](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html): Spark operations used here (~10 min).

## How to read this notebook

The analysis is written to be **reproducible** and followed by newcomers. Columns are loaded by explicit name, samples use fixed `RANDOM_STATE`, and each figure names the decision it drives.

> **Note:** a fact to pin down.
>
> **Tip:** a good habit.
>
> **Warning:** something that'd bias a downstream model if you miss it.
>
> **Go deeper:** curated background link.

Each figure has a **What you're seeing / Notice / Why it matters** caption. The "Why it matters" line states the concrete modeling decision the evidence drives.

## Environment check

Import the tools and set the house plotting theme (shared by every notebook).

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup

# The tools we lean on across the project.
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# House plotting theme: clean grid + colorblind-safe palette, used everywhere.
sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

# The one true seed, so any sample below is reproducible run-to-run.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("seaborn:", sns.__version__)

## Load the data

The ETL writes two Parquet variants: full dataset (~16.5M rows) and narrative-only subset (smaller, faster). This notebook prefers the narrative-only file. Exact class balance over all complaints is in `02_eda_supervised.ipynb`.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

# Prefer the smaller narrative-only file for a fast look; fall back to the full one.
# Resolve the data directory: the UC volume when running on Databricks,
# else the local ../data produced by `make parquet`.
VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
candidates = [data_dir / "complaints_narrative_only.parquet", data_dir / "complaints.parquet"]
path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError(
        "No Parquet found in data/. Run `make parquet` (or `make parquet NARRATIVE_ONLY=1`) first."
    )
print("Loading:", path.name)
df = spark.read.parquet(str(path))
n_rows = df.count()
print(f"Loaded {n_rows:,} rows x {len(df.columns)} columns")

## First pass: peek, shape, types, summary

Know the data's shape, types, and holes before trusting any pattern. Spark computes shape and summary. Only small results pull into pandas to display.

> **Go deeper:**
> - [PySpark: DataFrame.describe](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.describe.html): per-column numeric summary Spark computes on the full dataset (~3 min).

In [ ]:
print("First 5 rows:")
display(df.limit(5).toPandas())

print("\nDataset shape (rows, columns):", (n_rows, len(df.columns)))
print("\nColumn names and data types:")
print(df.dtypes)
print("\nDescriptive statistics (numeric columns):")
display(df.describe().toPandas())

> **Note:** the ETL emits an 18-column snake_case schema (`include/complaintiq/schema.h`). Raw CFPB names (`Product`, ...) mean a stale dump. Regenerate with `make parquet`.

## Structure and missing values

Spark aggregates nulls per column. The result pulls into pandas to print. Many CFPB fields are sparse.

> **Go deeper:**
> - [PySpark: handling missing data](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.na.html): how Spark represents nulls and options beyond dropping rows (~8 min).

In [ ]:
print("Structure:")
df.printSchema()
print("\nMissing values by column:")
missing = (
    df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).toPandas().T[0]
)
print(missing)

## A first glimpse of the target

`monetary_relief` is the ETL-derived label: `1` when the complaint closed with monetary relief, else `0`. Expect strong imbalance.

In [ ]:
if "monetary_relief" not in df.columns:
    raise KeyError(
        "monetary_relief not found - this looks like a stale raw-schema Parquet. "
        "Regenerate with `make parquet` (contract: include/complaintiq/schema.h)."
    )
print("monetary_relief value counts:")
counts = (
    df.groupBy("monetary_relief")
    .count()
    .orderBy("monetary_relief")
    .toPandas()
    .set_index("monetary_relief")["count"]
)
print(counts)
pos_rate = df.agg(F.mean("monetary_relief")).first()[0]
print(f"\npositive rate: {pos_rate:.4%}")

> **What you're seeing:** per-class counts and relief fraction.
>
> **Notice:** the positive class is tiny (about 1.28% on July 2026, roughly 1 in 78).
>
> **Why it matters:** confirms heavy imbalance. `02` will use imbalance-aware metrics, not accuracy. At ~1.28% base rate, a top-ranked queue is only worth building if its lift is large.

## Next

Environment and data check out. Continue in **`02_eda_supervised.ipynb`** and **`03_eda_unsupervised.ipynb`**. They share data and then diverge by task.